# Chapter 11 — Hard Negatives

**Book alignment:** Embeddings From First Principles, Chapter 11

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** A benchmark's difficulty is set by its *negatives*.
Does the ranking margin — `cos(query, correct) − cos(query, hardest negative)` — collapse
as the negatives get harder, from a comfortable +0.47 against random negatives to +0.03
against a role-reversed `relation-swap` (the committed Wave 1 measurement)?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Reproduce margin collapse: random vs topic-matched negatives

In [ ]:
D = 64
def unit(v): return v / np.linalg.norm(v)

topics = [rng.standard_normal(D) for _ in range(20)]
def item(topic, jitter=0.2):
    return unit(topic + rng.standard_normal(D) * jitter)

margins_random, margins_topic = [], []
for t in topics:
    q, correct = item(t, 0.15), item(t, 0.15)
    random_neg = unit(rng.standard_normal(D))
    topic_neg  = item(t, 0.15)                      # same topic, "different claim"
    margins_random.append(float(q @ correct - q @ random_neg))
    margins_topic.append(float(q @ correct - q @ topic_neg))

print(f"mean margin vs random negatives : {np.mean(margins_random):+.3f}")
print(f"mean margin vs topic negatives  : {np.mean(margins_topic):+.3f}")
assert np.mean(margins_random) > np.mean(margins_topic) + 0.2
print("the comfortable margin was almost entirely 'this passage is not about the topic'")

## 2. The measured collapse on RELATE (Wave 1)

In [ ]:
mc = art("wave1", "margin-collapse.json")
ns = mc["negative_sets"]
for name in ("random", "in_model", "structured_perturbation", "lexical"):
    v = ns[name]
    print(f"  {name:24} mean margin {v['mean_margin']:+.4f}   Recall@1 {v['recall_at_1_vs_negset']:.3f}")
by_rel = mc["structured_margin_by_relation"]
print("\n  structured margin by relation:", {k: round(v, 3) for k, v in by_rel.items()})

assert ns["random"]["mean_margin"] > 0.45
assert ns["lexical"]["mean_margin"] < 0.07                 # BM25-matched negatives: 5-8x collapse
assert by_rel["relation-swap"] < 0.03                      # role reversal: present but operationally gone
print("\n+0.47 -> +0.06 (lexical) -> +0.03 (relation-swap): resolution for the distinction that matters is at the noise floor")

## What we earned

Easy negatives measure topical separation; hard negatives measure the distinction you
actually care about. On RELATE the +0.47 random-negative margin fell to +0.06 against
BM25-matched negatives and +0.03 against a role reversal — present, but operationally gone,
which is exactly where calibration (Chapter 14) fails. Any retrieval number must state the
negative distribution it was measured against.

**Notebook 12 / Chapter 12** zooms out: retrieval is an eight-stage policy, and the policy —
not the model — is the artifact you version.